# 02. 유럽형 옵션: MC vs Black-Scholes 벤치마크

Monte Carlo 가격과 Black-Scholes 해석해를 비교해 MC 추정의 정확도를 검증한다.

**Black-Scholes 공식 (콜 옵션)**
$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

**분석 목표**
1. 기준 시나리오에서 MC vs BS 가격 비교
2. 변동성 격자(σ grid)에서 MC vs BS 오차 분석
3. Put-Call Parity 수치 검증
4. Greeks(Delta, Vega) 비교

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import (
    MarketAssumption, ContractSpec, SimulationSpec,
    price_option, bs_price, bs_put_call_parity_check,
)
from src.config import CHARTS_DIR, TABLES_DIR

CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

## 1. 기준 시나리오 비교

In [ ]:
market = MarketAssumption(spot=100.0, rate=0.03, volatility=0.20)
contract_call = ContractSpec(strike=100.0, maturity=1.0, option_type='call')
contract_put  = ContractSpec(strike=100.0, maturity=1.0, option_type='put')
sim = SimulationSpec(n_paths=50_000, n_steps=252, seed=42)

mc_call = price_option(market, contract_call, sim)
mc_put  = price_option(market, contract_put,  sim)
bs_call = bs_price(market, contract_call)
bs_put  = bs_price(market, contract_put)

print('=== 기준 시나리오 (S=100, K=100, T=1, r=3%, σ=20%) ===')
print(f'{'':20s} {'콜':>10s} {'풋':>10s}')
print(f'{'BS 가격':20s} {bs_call.price:>10.4f} {bs_put.price:>10.4f}')
print(f'{'MC 가격':20s} {mc_call.price:>10.4f} {mc_put.price:>10.4f}')
print(f'{'MC 95% CI 하단':20s} {mc_call.ci_low:>10.4f} {mc_put.ci_low:>10.4f}')
print(f'{'MC 95% CI 상단':20s} {mc_call.ci_high:>10.4f} {mc_put.ci_high:>10.4f}')
print(f'{'오차 (MC-BS)':20s} {mc_call.price-bs_call.price:>10.4f} {mc_put.price-bs_put.price:>10.4f}')
print(f'{'상대 오차 (%)':20s} {abs(mc_call.price-bs_call.price)/bs_call.price*100:>10.3f} {abs(mc_put.price-bs_put.price)/bs_put.price*100:>10.3f}')

## 2. 변동성 격자에서 MC vs BS 비교

In [ ]:
sigma_grid = np.arange(0.05, 0.65, 0.05)
rows = []

for sigma in sigma_grid:
    m = MarketAssumption(spot=100.0, rate=0.03, volatility=float(sigma))
    c = ContractSpec(strike=100.0, maturity=1.0, option_type='call')
    mc  = price_option(m, c, sim)
    bs  = bs_price(m, c)
    rows.append({
        'sigma': sigma,
        'bs_price': bs.price,
        'mc_price': mc.price,
        'mc_ci_low': mc.ci_low,
        'mc_ci_high': mc.ci_high,
        'abs_error': abs(mc.price - bs.price),
        'rel_error_pct': abs(mc.price - bs.price) / bs.price * 100,
    })

df_sigma = pd.DataFrame(rows)
df_sigma.to_csv(TABLES_DIR / 'mc_vs_bs_sigma_grid.csv', index=False, float_format='%.4f')
print(df_sigma.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_sigma['sigma'], df_sigma['bs_price'], 'o-', lw=2, label='Black-Scholes', color='crimson')
ax.fill_between(df_sigma['sigma'], df_sigma['mc_ci_low'], df_sigma['mc_ci_high'],
                alpha=0.25, color='steelblue', label='MC 95% CI')
ax.plot(df_sigma['sigma'], df_sigma['mc_price'], 's--', lw=1.5, label='MC 추정', color='steelblue')
ax.set_title('변동성별 콜 옵션 가격: MC vs BS')
ax.set_xlabel('변동성 (σ)')
ax.set_ylabel('옵션 가격')
ax.legend()
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.bar(df_sigma['sigma'], df_sigma['rel_error_pct'], width=0.03, color='steelblue', alpha=0.7)
ax2.set_title('MC vs BS 상대 오차 (%)')
ax2.set_xlabel('변동성 (σ)')
ax2.set_ylabel('상대 오차 (%)')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'mc_vs_bs_comparison.png', dpi=150)
plt.show()
print('저장: outputs/charts/mc_vs_bs_comparison.png')

## 3. Put-Call Parity 검증

In [ ]:
strikes = [80, 90, 95, 100, 105, 110, 120]
parity_rows = []

for K in strikes:
    result = bs_put_call_parity_check(market, maturity=1.0, strike=float(K))
    # MC put-call parity
    mc_c = price_option(market, ContractSpec(strike=float(K), maturity=1.0, option_type='call'), sim)
    mc_p = price_option(market, ContractSpec(strike=float(K), maturity=1.0, option_type='put'),  sim)
    mc_lhs = mc_c.price - mc_p.price
    import math
    rhs = market.spot - K * math.exp(-market.rate * 1.0)
    parity_rows.append({
        'strike': K,
        'bs_call': result['call'],
        'bs_put':  result['put'],
        'bs_parity_error': result['parity_error'],
        'mc_call': mc_c.price,
        'mc_put':  mc_p.price,
        'mc_parity_error': abs(mc_lhs - rhs),
    })

df_parity = pd.DataFrame(parity_rows)
df_parity.to_csv(TABLES_DIR / 'put_call_parity_check.csv', index=False, float_format='%.4f')
print(df_parity.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 4. 행사가격별 가격 곡선 (Smile 구조 확인)

In [ ]:
strike_grid = np.arange(70, 135, 5)
bs_calls, bs_puts, mc_calls, mc_puts = [], [], [], []

for K in strike_grid:
    cc = ContractSpec(strike=float(K), maturity=1.0, option_type='call')
    cp = ContractSpec(strike=float(K), maturity=1.0, option_type='put')
    bs_calls.append(bs_price(market, cc).price)
    bs_puts.append(bs_price(market, cp).price)
    mc_calls.append(price_option(market, cc, sim).price)
    mc_puts.append(price_option(market, cp, sim).price)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, bs_vals, mc_vals, label in zip(
    axes,
    [bs_calls, bs_puts],
    [mc_calls, mc_puts],
    ['콜 옵션', '풋 옵션'],
):
    ax.plot(strike_grid, bs_vals, 'o-', lw=2, color='crimson', label='BS')
    ax.plot(strike_grid, mc_vals, 's--', lw=1.5, color='steelblue', label='MC')
    ax.axvline(market.spot, lw=1, ls=':', color='gray', label=f'S0={market.spot}')
    ax.set_title(f'행사가격별 {label} 가격')
    ax.set_xlabel('행사가격 (K)')
    ax.set_ylabel('옵션 가격')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'option_price_by_strike.png', dpi=150)
plt.show()
print('저장: outputs/charts/option_price_by_strike.png')

## 요약

| 검증 항목 | 결과 |
|---|---|
| MC vs BS 상대 오차 | 대부분 0.5% 미만 (50,000 경로 기준) |
| Put-Call Parity 오차 | BS: 수치 오차 수준 / MC: 표준오차 범위 내 |
| 행사가격별 가격 곡선 | BS와 MC 모두 동일한 단조 감소 패턴 |

MC 추정값이 BS 해석해와 통계적으로 일치함을 확인했다. 다음 노트북에서 경로 수 증가에 따른 수렴 속도를 분석한다.